<a href="https://colab.research.google.com/github/Santiago-Soria/proyecto-transformacion-texto-imagen/blob/sant_branch/notebooks/3.2_run_experimentos_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Clonar repositorio
!git clone https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen.git


Cloning into 'proyecto-transformacion-texto-imagen'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 211 (delta 90), reused 152 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 3.07 MiB | 28.83 MiB/s, done.
Resolving deltas: 100% (90/90), done.


In [14]:
# Cambiar al directorio del proyecto
%cd proyecto-transformacion-texto-imagen

/content/proyecto-transformacion-texto-imagen/proyecto-transformacion-texto-imagen


In [15]:
!git fetch
!git checkout sant_branch
!git pull origin sant_branch

Branch 'sant_branch' set up to track remote branch 'sant_branch' from 'origin'.
Switched to a new branch 'sant_branch'
From https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen
 * branch            sant_branch -> FETCH_HEAD
Already up to date.


In [16]:
!git branch

  main
* sant_branch


In [7]:
# Instalar dependencias del proyecto
!pip install transformers datasets torch scikit-learn pandas numpy spacy importlib matplotlib-venn
!python -m spacy download es_core_news_sm

  Preparing metadata (setup.py) ... done
  Created wheel for importlib: filename=importlib-1.0.4-py3-none-any.whl size=5850 sha256=7c0673fe1b5b7542477a490bce2d200845c73a8c9505627a47ab20a8aa339d9f
  Stored in directory: /root/.cache/pip/wheels/40/41/c4/d925a53b7b7e75a65369e1b17f7bade00d7907ac5a7d74dc5f
Successfully built importlib
  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [17]:
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"Dispositivo: {torch.cuda.get_device_name(0)}")

GPU disponible: True
Dispositivo: Tesla T4


In [18]:
import sys
import os
import importlib
# Agregar src al path desde la raíz del proyecto
sys.path.insert(0, '/content/proyecto-transformacion-texto-imagen/src')

from models import train_classic
# Recargar el módulo
importlib.reload(train_classic)

# Ahora importar normalmente
import pandas as pd
from preprocessing_utils import preprocess_text
from features.extraction import FeatureExtractor
from models.train_classic import train_logistic
from models.train_transformer import run_finetuning

In [19]:
from re import X
train = pd.read_csv('/content/proyecto-transformacion-texto-imagen/data/processed/train.csv')
test = pd.read_csv('/content/proyecto-transformacion-texto-imagen/data/processed/test.csv')

# Limpieza Mínima en el Preprocesamiento
X_train_p1 = [preprocess_text(t, 'P1') for t in train['text']]
X_test_p1 = [preprocess_text(t, 'P1') for t in test['text']]

# Extractor de Caractetísticas
extractor =  FeatureExtractor()

### **Experimento 2.2:** Limpieza Básica + RoBERTa-BNE (Frozen) + Regresión Logística

In [21]:
import inspect
print(inspect.getsource(train_logistic))

def train_logistic(X_train, y_train, X_test, y_test, experiment_name="exp_generic", models_dir=None):
    print(f"Entrenando Regresión Logística para {experiment_name}...")
    
    # Entrenamiento
    clf = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=42)
    clf.fit(X_train, y_train)
    
    # Evaluación
    preds = clf.predict(X_test)
    score = f1_score(y_test, preds, average='macro')
    
    # Reporte
    print(f"\n---> Resultado (Macro-F1) - {experiment_name}: {score:.4f}")
    print(f"--- Resultados {experiment_name} ---")
    print(classification_report(y_test, preds))

    # Guardar modelo - Compatible con local y Colab
    if models_dir is None:
        # Intentar detectar el entorno
        try:
            # Si estamos en un archivo .py (local)
            base_dir = os.path.dirname(os.path.abspath(__file__))
            models_dir = os.path.join(base_dir, "..", "..", "models")
        except NameError:
            # Si estamos en notebook

In [ ]:
# ==========================================
# EXP 2.2: Frozen RoBERTa + LR
# ==========================================
# Usamos P1 (Transformers prefieren texto con estructura, no P2)
print("Generando embeddings. . .")
X_train_emb = extractor.get_transformer_embeddings(X_train_p1, "bertin-project/bertin-roberta-base-spanish")
X_test_emb = extractor.get_transformer_embeddings(X_test_p1, "bertin-project/bertin-roberta-base-spanish")

models_dir = '/content/proyecto-transformacion-texto-imagen/models'
os.makedirs(models_dir, exist_ok=True)

train_logistic(X_train_emb, train['manual_classification'],
               X_test_emb, test['manual_classification'], "Exp_2.2_RoBERTa_Frozen")

Generando embeddings. . .
--> Extrayendo embeddings con bertin-project/bertin-roberta-base-spanish...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 29/29 [00:05<00:00,  4.94it/s]


--> Extrayendo embeddings con bertin-project/bertin-roberta-base-spanish...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 4/4 [00:00<00:00,  4.84it/s]


Entrenando Regresión Logística para Exp_2.2_RoBERTa_Frozen...

---> Resultado (Macro-F1) - Exp_2.2_RoBERTa_Frozen: 0.6096
--- Resultados Exp_2.2_RoBERTa_Frozen ---
              precision    recall  f1-score   support

           0       0.71      0.66      0.68        70
           1       0.51      0.57      0.54        44

    accuracy                           0.62       114
   macro avg       0.61      0.61      0.61       114
weighted avg       0.63      0.62      0.63       114

✓ Modelo guardado en: /content/proyecto-transformacion-texto-imagen/src/models/../../models/Exp_2.2_RoBERTa_Frozen.pkl


(LogisticRegression(class_weight='balanced', random_state=42, solver='liblinear'),
 array([0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
        1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1,
        1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0,
        0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0,
        1, 0, 1, 1]),
 0.609557945041816)

### **Experimento 2.3:** Limpieza Básica + mDeBERTa-v3-base (Frozen) + Regresión Logística

In [ ]:
# ==========================================
# EXP 2.3: Frozen mDeBERTa-v3-base + LR
# ==========================================
# Usamos P1 (Transformers prefieren texto con estructura, no P2)
print("Generando embeddings. . .")
X_train_emb = extractor.get_transformer_embeddings(X_train_p1, "microsoft/mdeberta-v3-base")
X_test_emb = extractor.get_transformer_embeddings(X_test_p1, "microsoft/mdeberta-v3-base")

models_dir = '/content/proyecto-transformacion-texto-imagen/models'
os.makedirs(models_dir, exist_ok=True)

train_logistic(X_train_emb, train['manual_classification'],
               X_test_emb, test['manual_classification'], "Exp_2.3_mDeBERTa_Frozen")

Generando embeddings. . .
--> Extrayendo embeddings con microsoft/mdeberta-v3-base...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

...

--> Extrayendo embeddings con microsoft/mdeberta-v3-base...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/ar

Entrenando Regresión Logística para Exp_2.3_mDeBERTa_Frozen...

---> Resultado (Macro-F1) - Exp_2.3_mDeBERTa_Frozen: 0.5260
--- Resultados Exp_2.3_mDeBERTa_Frozen ---
              precision    recall  f1-score   support

           0       0.63      0.64      0.64        70
           1       0.42      0.41      0.41        44

    accuracy                           0.55       114
   macro avg       0.53      0.53      0.53       114
weighted avg       0.55      0.55      0.55       114

✓ Modelo guardado en: /content/proyecto-transformacion-texto-imagen/src/models/../../models/Exp_2.3_mDeBERTa_Frozen.pkl


(LogisticRegression(class_weight='balanced', random_state=42, solver='liblinear'),
 array([1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1,
        0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0,
        0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0,
        1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
        1, 0, 0, 1]),
 0.5260454878943507)

### **Experimento 3.2:** Limpieza Básica + RoBERTa-BNE(Fine-Tuning) + Softmax

In [11]:
import inspect
print(inspect.getsource(run_finetuning))

def run_finetuning(train_texts, train_labels, val_texts, val_labels, model_name="PlanTL-GOB-ES/roberta-base-bne"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    
    train_ds = DepressionDataset(train_texts, train_labels, tokenizer)
    val_ds = DepressionDataset(val_texts, val_labels, tokenizer)
    safe_model_name = model_name.replace("/", "_")
    
    args = TrainingArguments(
        output_dir=f"models/checkpoints/{safe_model_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        num_train_epochs=4,
        weight_decay=0.01,
        load_best_model_at_end=True,
    )
    
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
    )
    
    trainer.train()
    return trainer



In [12]:
# ==========================================
# EXP 3.2: Fine-Tuning RoBERTa
# ==========================================
# Este es el pesado. Asegúrate de estar en GPU.
trainer = run_finetuning(X_train_p1, train['manual_classification'].values,
                         X_test_p1, test['manual_classification'].values,
                         "bertin-project/bertin-roberta-base-spanish")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye